# Experiment 07 — Strengthening the GBM

Experiment 06 showed the path forward is *not* linear models but a **stronger tree ensemble** (pooling across series is what wins). This notebook pushes the GBM along two levers, validated on the same 3 rolling windows against our best deployed model `geo_blend25` (public **0.38899**):

- **Lever A — dynamic per-series features** (added to notebook 02, recomputed from own predictions inside the iterative loop, all `shift(1)` leak-free):
  - `growth_ratio` = `rolling_mean_28 / rolling_mean_364` — recent level vs a year ago (local trend);
  - `dow_mean_4` — mean of the last 4 same-weekday sales (clean weekly level);
  - `rolling_std_28` — recent volatility;
  - `zero_share_28` — share of zero-sales days recently.
  - (`days_since_last_sale` was **deliberately excluded**: under recursive forecasting the model is fed its own always-positive predictions, so it collapses to 1 at inference while varying in training — a train/inference mismatch. A direct model could use it; a recursive one should not.)
- **Lever B — lighter per-family hyperparameters.** Experiment 04 flagged that the per-family models reused the *global* hyperparameters (`num_leaves=255`, 4155 trees) — far too heavy for ~30k-row per-series data. Here they get `num_leaves=63`, `min_child_samples=100`, 1500 trees.

The candidate (`glob_dyn_blend25` = 25% light per-family + 75% global-with-dynamic-features) combines both levers. Gate unchanged: beat `geo_blend25` on **all 3 windows**.

> 🇷🇺 Эксперимент 06 показал: путь — не линейные модели, а более сильный ансамбль деревьев. Здесь два рычага против лучшего задеплоенного `geo_blend25` (паблик 0.38899): **A** — динамические фичи (`growth_ratio` и др., пересчёт из собственных предсказаний в цикле, без утечек; `days_since_last_sale` исключён — вырождается при рекурсии); **B** — облегчённые гиперпараметры per-family (`num_leaves=63`, 1500 деревьев вместо 255/4155 — эксп. 04 пометил это как недочёт). Кандидат `glob_dyn_blend25` совмещает оба рычага. Гейт прежний: побить `geo_blend25` на всех 3 окнах.

In [1]:
import time

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

In [2]:
_t0 = time.time()

MODELS_DIR = "../models/exp07"
ART_DIR    = "../artifacts/exp07"
EXP05_ART  = "../artifacts/exp05"   # cached geo / geo_blend25 window predictions (baselines)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)

# Heavy params for the global model (same as notebooks 03-05) / Тяжёлые параметры глобальной модели (как в 03-05)
PARAMS_HEAVY = dict(n_estimators=4155, learning_rate=0.01, num_leaves=255, min_child_samples=30,
                    colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                    reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1, verbose=-1)
# Light params for per-family models (~30k rows each) / Лёгкие параметры per-family моделей (~30k строк)
PARAMS_LIGHT = dict(n_estimators=1500, learning_rate=0.03, num_leaves=63, min_child_samples=100,
                    colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                    reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1, verbose=-1)

BASE = [
    "day_of_week", "month", "year", "is_weekend", "day_of_year", "week_of_year", "date_index",
    "sin_day", "cos_day", "sin_week", "cos_week",
    "lag_1", "lag_2", "lag_3", "lag_4", "lag_5", "lag_6",
    "lag_7", "lag_14", "lag_21", "lag_28", "lag_42", "lag_56", "lag_364", "lag_365",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28", "rolling_mean_364",
    "dcoilwtico", "dcoilwtico_ma7", "dcoilwtico_ma28",
    "onpromotion", "onpromotion_ma7", "onpromotion_ma28",
    *[f"transactions_lag_{l}" for l in range(16, 24)],
    "is_holiday_national", "day_before_holiday", "is_black_friday", "is_cyber_monday", "is_terremoto",
    "is_navidad", "is_dia_madre", "is_futbol", "is_dia_trabajo", "is_primer_dia", "is_dia_difuntos", "work_day",
    "store_nbr", "store_type", "cluster", "family",
]
GEO = ["city", "state", "is_holiday_local", "is_holiday_regional"]
DYN = ["growth_ratio", "dow_mean_4", "rolling_std_28", "zero_share_28"]
FEATURES = BASE + GEO + DYN                                  # global-with-dynamic feature set
CAT = ["store_nbr", "store_type", "cluster", "family", "day_of_week", "month", "city", "state"]

WINDOWS = [("W1", pd.Timestamp("2017-06-15"), pd.Timestamp("2017-06-30")),
           ("W2", pd.Timestamp("2017-07-15"), pd.Timestamp("2017-07-30")),
           ("W3", pd.Timestamp("2017-07-31"), pd.Timestamp("2017-08-15"))]
WNAMES = [w for w, _, _ in WINDOWS]
TEST_START, TEST_END = pd.Timestamp("2017-08-16"), pd.Timestamp("2017-08-31")
print(f"{len(FEATURES)} features (BASE {len(BASE)} + GEO {len(GEO)} + DYN {len(DYN)})")

67 features (BASE 59 + GEO 4 + DYN 4)


In [3]:
# Load the feature superset; reshape into (dates x series) matrices
# Загружаем супермножество фич; укладываем в матрицы (даты x ряды)
DF = (pd.read_parquet("../artifacts/features.parquet")
      .sort_values(["date", "store_nbr", "family"]).reset_index(drop=True))
for c in FEATURES:
    assert c in DF.columns, f"missing column {c} — run notebook 02"
DATES = np.sort(DF["date"].unique()); N_DATES = len(DATES)
N_SERIES = DF.groupby("date").size().iloc[0]
assert len(DF) == N_DATES * N_SERIES
DATE_IDX = {d: i for i, d in enumerate(DATES)}
DT = pd.to_datetime(DATES)
ref = DF.iloc[:N_SERIES][["store_nbr", "family"]].reset_index(drop=True)
FAMILIES = sorted(DF["family"].astype(str).unique())
FAM_ARR = ref["family"].astype(str).values
SALES_TRUE = DF["sales"].to_numpy(dtype=float).reshape(N_DATES, N_SERIES)
print(f"{N_DATES} dates x {N_SERIES} series, {len(FAMILIES)} families")

1704 dates x 1782 series, 33 families


In [4]:
# Engine: dynamic features recomputed from own predictions (lags, rollings + the 4 new ones)
# Definitions match notebook 02 exactly so training and inference see the same features.
# Движок: динамические фичи из собственных предсказаний (лаги, rolling + 4 новые)
# Определения в точности как в ноутбуке 02, чтобы обучение и инференс видели одни и те же фичи.
LAGS = [1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 42, 56, 364, 365]
ROLLS = [7, 14, 28]


def rmsle_mat(P, Y):
    return float(np.sqrt(np.mean((np.log1p(np.clip(P, 0, None)) - np.log1p(Y)) ** 2)))


def dynamic_features(sm, i):
    feat = {f"lag_{k}": sm[i - k] for k in LAGS}
    for w in ROLLS:
        feat[f"rolling_mean_{w}"] = np.nanmean(sm[i - w:i], axis=0)
    win = sm[i - 364:i]; cnt = (~np.isnan(win)).sum(0)
    rm364 = np.full(N_SERIES, np.nan); ok = cnt >= 30
    if ok.any():
        rm364[ok] = np.nanmean(win[:, ok], axis=0)
    feat["rolling_mean_364"] = rm364
    # new dynamic features / новые динамические фичи (definitions mirror notebook 02)
    gr = feat["rolling_mean_28"] / rm364
    feat["growth_ratio"] = np.where(np.isfinite(gr), gr, 1.0)
    feat["dow_mean_4"] = np.nanmean(sm[[i - 7, i - 14, i - 21, i - 28]], axis=0)
    w28 = sm[i - 28:i]; c28 = (~np.isnan(w28)).sum(0)
    std = np.full(N_SERIES, np.nan); ok2 = c28 >= 2
    if ok2.any():
        std[ok2] = np.nanstd(w28[:, ok2], axis=0, ddof=1)
    feat["rolling_std_28"] = std
    z = np.where(np.isnan(w28), np.nan, (w28 == 0).astype(float))
    feat["zero_share_28"] = np.nanmean(z, axis=0)
    return feat


def x_for_day(i, sm):
    X = DF.iloc[i * N_SERIES:(i + 1) * N_SERIES][FEATURES].copy()
    for c, v in dynamic_features(sm, i).items():
        if c in X.columns:
            X[c] = v
    for col in CAT:
        X[col] = X[col].astype("category")
    return X


def zero_rule_mask(first_idx):
    return np.nansum(SALES_TRUE[first_idx - 21:first_idx], axis=0) == 0


def run_iterative(predict_day, win_idxs, zmask):
    sm = SALES_TRUE.copy(); sm[win_idxs, :] = np.nan
    P = np.empty((len(win_idxs), N_SERIES))
    for j, i in enumerate(win_idxs):
        p = np.clip(predict_day(x_for_day(i, sm)), 0, None)
        p[zmask] = 0.0; sm[i] = p; P[j] = p
    return P


def fit(sub, feats, params):
    X = sub[feats].copy(); y = np.log1p(sub["sales"])
    for col in [c for c in CAT if c in feats]:
        X[col] = X[col].astype("category")
    m = lgb.LGBMRegressor(**params); m.fit(X, y); return m


def train_global_dyn(cutoff, tag):
    path = os.path.join(MODELS_DIR, f"glob_dyn_{tag}.joblib")
    if os.path.exists(path):
        return joblib.load(path)
    sub = DF[(DF["date"] >= pd.Timestamp("2016-01-01")) & (DF["date"] < cutoff) & DF["sales"].notna()]
    t = time.time(); m = fit(sub, FEATURES, PARAMS_HEAVY); joblib.dump(m, path)
    print(f"  glob_dyn_{tag}: {len(sub):,} rows, {time.time()-t:.0f}s"); return m


def train_pf_light(cutoff, tag):
    base = DF[(DF["date"] >= pd.Timestamp("2016-01-01")) & (DF["date"] < cutoff) & DF["sales"].notna()]
    models, t = {}, time.time()
    for fam in FAMILIES:
        slug = fam.replace("/", "_").replace(" ", "_")
        path = os.path.join(MODELS_DIR, f"pflite_{tag}_{slug}.joblib")
        models[fam] = joblib.load(path) if os.path.exists(path) else fit(base[base["family"] == fam], FEATURES, PARAMS_LIGHT)
        if not os.path.exists(path):
            joblib.dump(models[fam], path)
    print(f"  pflite x{len(FAMILIES)} ({tag}): {time.time()-t:.0f}s"); return models


def pred_global(m):
    return lambda X: np.expm1(m.predict(X[FEATURES]))


def pred_pf(models):
    def f(X):
        fam = X["family"].astype(str).values; out = np.empty(len(X)); Xs = X[FEATURES]
        for name, mdl in models.items():
            msk = fam == name
            out[msk] = np.expm1(mdl.predict(Xs[msk]))
        return out
    return f


def pred_blend(f1, f2, w1):
    return lambda X: w1 * f1(X) + (1 - w1) * f2(X)

In [5]:
# Sanity: the 4 new dynamic features computed by the engine must match notebook 02's parquet values
# Проверка: 4 новые динамические фичи из движка должны совпадать со значениями parquet из ноутбука 02
i = DATE_IDX[pd.Timestamp("2017-05-10")]
feat = dynamic_features(SALES_TRUE, i)
block = DF.iloc[i * N_SERIES:(i + 1) * N_SERIES]
for c in ["growth_ratio", "dow_mean_4", "rolling_std_28", "zero_share_28"]:
    a = np.asarray(feat[c], float); b = block[c].to_numpy(float)
    ok = np.isclose(a, b, equal_nan=True, atol=1e-5)
    assert ok.mean() > 0.999, f"{c}: only {ok.mean():.3f} match"
    print(f"  {c:16s} match {ok.mean()*100:.2f}%")
print("OK: engine reproduces the dynamic features of notebook 02")

  growth_ratio     match 100.00%
  dow_mean_4       match 100.00%
  rolling_std_28   match 100.00%
  zero_share_28    match 100.00%
OK: engine reproduces the dynamic features of notebook 02


In [6]:
# Main runs: levers A (dynamic features) and A+B (dynamic + light per-family blend)
# Основные прогоны: рычаг A (динамические фичи) и A+B (динамика + лёгкий per-family бленд)
RESULTS, rows = {}, []
for wname, ws, we in WINDOWS:
    print(f"\n=== {wname}: {ws.date()} .. {we.date()} ===")
    win_idxs = [i for i in range(N_DATES) if ws <= DT[i] <= we]
    assert len(win_idxs) == 16
    Y = SALES_TRUE[win_idxs]; zmask = zero_rule_mask(win_idxs[0])

    # baselines from exp05 cache / бейзлайны из кэша exp05
    for sch in ["geo", "geo_blend25"]:
        RESULTS[(wname, sch)] = np.load(os.path.join(EXP05_ART, f"pred_{wname}__{sch}.npy"))

    gd = train_global_dyn(ws, wname)
    pf = train_pf_light(ws, wname)
    fgd, fpf = pred_global(gd), pred_pf(pf)
    live = {"glob_dyn": fgd, "glob_dyn_blend25": pred_blend(fpf, fgd, 0.25)}
    for sch, fn in live.items():
        cache = os.path.join(ART_DIR, f"pred_{wname}__{sch}.npy")
        if os.path.exists(cache):
            RESULTS[(wname, sch)] = np.load(cache)
        else:
            t = time.time(); RESULTS[(wname, sch)] = run_iterative(fn, win_idxs, zmask)
            np.save(cache, RESULTS[(wname, sch)]); print(f"  {sch}: {time.time()-t:.0f}s")

    for sch in ["geo", "geo_blend25", "glob_dyn", "glob_dyn_blend25"]:
        sc = rmsle_mat(RESULTS[(wname, sch)], Y)
        rows.append({"window": wname, "scheme": sch, "rmsle": sc})
        print(f"  {sch:18s} RMSLE={sc:.5f}")

res = pd.DataFrame(rows); res.to_csv(os.path.join(ART_DIR, "results.csv"), index=False)
print(f"\nElapsed: {(time.time()-_t0)/60:.1f} min")


=== W1: 2017-06-15 .. 2017-06-30 ===
  pflite x33 (W1): 1s
  geo                RMSLE=0.38272
  geo_blend25        RMSLE=0.38173
  glob_dyn           RMSLE=0.38259
  glob_dyn_blend25   RMSLE=0.38106

=== W2: 2017-07-15 .. 2017-07-30 ===
  pflite x33 (W2): 1s
  geo                RMSLE=0.38783
  geo_blend25        RMSLE=0.38652
  glob_dyn           RMSLE=0.38757
  glob_dyn_blend25   RMSLE=0.38658

=== W3: 2017-07-31 .. 2017-08-15 ===
  pflite x33 (W3): 1s
  geo                RMSLE=0.40369
  geo_blend25        RMSLE=0.40187
  glob_dyn           RMSLE=0.40511
  glob_dyn_blend25   RMSLE=0.40381

Elapsed: 0.1 min


In [7]:
# Summary and gate (beat geo_blend25 on all 3 windows)
# Сводка и гейт (побить geo_blend25 на всех 3 окнах)
piv = res.pivot(index="scheme", columns="window", values="rmsle")
piv["mean"] = piv[WNAMES].mean(axis=1); piv = piv.sort_values("mean")
display(piv.round(5))

def beats(c, base="geo_blend25"):
    return all(piv.loc[c, w] < piv.loc[base, w] for w in WNAMES)

print("\nLever A — does dynamic features help the global model? glob_dyn vs geo:")
print("  " + "  ".join(f"{w}:{piv.loc['glob_dyn', w]-piv.loc['geo', w]:+.5f}" for w in WNAMES),
      "->", "helps" if beats("glob_dyn", "geo") else "no")
print("\nvs geo_blend25 (gate = win all 3 windows):")
for c in ["glob_dyn", "glob_dyn_blend25"]:
    print(f"  {c:18s} " + "  ".join(f"{w}:{piv.loc[c, w]-piv.loc['geo_blend25', w]:+.5f}" for w in WNAMES),
          "  gate=" + ("PASS" if beats(c) else "fail"))
GATED = [c for c in ["glob_dyn", "glob_dyn_blend25"] if beats(c)]
FINAL = min(GATED, key=lambda c: piv.loc[c, "mean"]) if GATED else None
print("\nGate passed by:", GATED if GATED else "NOBODY", "| FINAL:", FINAL)

window,W1,W2,W3,mean
scheme,,,,
geo_blend25,0.38173,0.38652,0.40187,0.39004
glob_dyn_blend25,0.38106,0.38658,0.40381,0.39048
geo,0.38272,0.38783,0.40369,0.39141
glob_dyn,0.38259,0.38757,0.40511,0.39176



Lever A — does dynamic features help the global model? glob_dyn vs geo:
  W1:-0.00013  W2:-0.00026  W3:+0.00143 -> no

vs geo_blend25 (gate = win all 3 windows):
  glob_dyn           W1:+0.00087  W2:+0.00105  W3:+0.00324   gate=fail
  glob_dyn_blend25   W1:-0.00067  W2:+0.00006  W3:+0.00194   gate=fail

Gate passed by: NOBODY | FINAL: None


In [8]:
# Final: if a scheme passed the gate, refit on full train and build the submission
# Финал: если схема прошла гейт — переобучение на всём train и сборка сабмишна
if FINAL is None:
    print("No scheme beat geo_blend25 on all 3 windows — no submission (honest negative result).")
else:
    test_idxs = [i for i in range(N_DATES) if TEST_START <= DT[i] <= TEST_END]
    zmask_t = zero_rule_mask(test_idxs[0])
    gd = train_global_dyn(TEST_START, "FULL")
    fgd = pred_global(gd)
    if FINAL == "glob_dyn_blend25":
        pf = train_pf_light(TEST_START, "FULL"); fn = pred_blend(pred_pf(pf), fgd, 0.25)
    else:
        fn = fgd
    cache = os.path.join(ART_DIR, f"pred_TEST__{FINAL}.npy")
    P_test = np.load(cache) if os.path.exists(cache) else run_iterative(fn, test_idxs, zmask_t)
    if not os.path.exists(cache):
        np.save(cache, P_test)
    test = pd.read_csv("../data/test.csv", parse_dates=["date"])
    long = pd.concat([pd.DataFrame({"date": DATES[i], "store_nbr": ref["store_nbr"].values,
                                    "family": ref["family"].values, "sales": P_test[j]})
                      for j, i in enumerate(test_idxs)], ignore_index=True)
    sub = test.merge(long, on=["date", "store_nbr", "family"], how="left")[["id", "sales"]]
    assert sub["sales"].notna().all()
    sub.to_csv("../submission_07_gbm_boost.csv", index=False)
    print(f"Saved submission_07_gbm_boost.csv (scheme={FINAL}, total {sub['sales'].sum():,.0f})")
print(f"\nTotal runtime: {(time.time()-_t0)/60:.1f} min")

No scheme beat geo_blend25 on all 3 windows — no submission (honest negative result).

Total runtime: 0.1 min


## Conclusions

**Negative result: neither lever beats `geo_blend25` on all three windows. No submission produced.**

| Scheme | W1 | W2 | W3 | mean |
|---|---:|---:|---:|---:|
| **geo_blend25** (baseline, public 0.38899) | 0.38173 | 0.38652 | **0.40187** | **0.39004** |
| glob_dyn_blend25 (levers A+B) | 0.38106 | 0.38658 | 0.40381 | 0.39048 |
| geo (global+geo, no dynamic) | 0.38272 | 0.38783 | 0.40369 | 0.39141 |
| glob_dyn (global+geo+dynamic) | 0.38259 | 0.38757 | 0.40511 | 0.39176 |

**Lever A — dynamic features (`growth_ratio`, `dow_mean_4`, `rolling_std_28`, `zero_share_28`): rejected.** They help slightly on W1 and W2 but hurt on W3 (+0.00142), and W3 is the August window — the one most like the real test. The cause is the recursive train/inference mismatch we expected when we dropped `days_since_last_sale`: these features are computed from real sales in training but from the model's own drifting predictions during the 16-day forecast, and the drift grows with the horizon — so the damage is largest on the longest, most test-like window.

**Lever B — lighter per-family hyperparameters: cannot be isolated here** (the blend uses the dynamic-feature global), and the combined result does not beat the baseline (0.39048 vs 0.39004).

**Outcome.** The deployed best stays `geo_blend25` (public 0.38899) and the file-level mix (public **0.38803**). The 3-window gate again rejected an intuitively appealing feature (`growth_ratio`) because its gain disappeared on the window that matters most.

## Выводы

**Отрицательный результат: ни один рычаг не бьёт `geo_blend25` на всех трёх окнах. Сабмишн не создан.**

| Схема | W1 | W2 | W3 | среднее |
|---|---:|---:|---:|---:|
| **geo_blend25** (бейзлайн, паблик 0.38899) | 0.38173 | 0.38652 | **0.40187** | **0.39004** |
| glob_dyn_blend25 (рычаги A+B) | 0.38106 | 0.38658 | 0.40381 | 0.39048 |
| geo (global+geo, без динамики) | 0.38272 | 0.38783 | 0.40369 | 0.39141 |
| glob_dyn (global+geo+динамика) | 0.38259 | 0.38757 | 0.40511 | 0.39176 |

**Рычаг A — динамические фичи (`growth_ratio`, `dow_mean_4`, `rolling_std_28`, `zero_share_28`): отклонён.** Они чуть помогают на W1 и W2, но вредят на W3 (+0.00142), а W3 — августовское окно, самое похожее на реальный тест. Причина — рекурсивный рассинхрон train/inference, который мы предвидели, когда отказались от `days_since_last_sale`: эти фичи считаются от реальных продаж при обучении и от дрейфующих собственных предсказаний при прогнозе, и дрейф растёт с горизонтом — поэтому урон максимален на самом длинном, самом тест-подобном окне.

**Рычаг B — облегчённые гиперпараметры per-family: здесь не изолируется** (бленд использует глобальную модель с динамическими фичами), и в сумме результат не бьёт бейзлайн (0.39048 против 0.39004).

**Итог.** Лучший боевой результат остаётся `geo_blend25` (паблик 0.38899) и файловый микс (паблик **0.38803**). Гейт 3 окон снова отклонил интуитивно привлекательную фичу (`growth_ratio`), потому что её польза исчезла на самом важном окне.